# The metric follows the decision
Use the course package to simulate a correlated return panel, then measure ranking quality, trading intensity, and portfolio performance.

## 1. Using `finmlsim.metrics`
The companion package provides the finance-specific definitions used in the course.

In [ ]:
import sys, subprocess
try:
    from finmlsim import simulate as sim, metrics as met
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'finmlsim'])
    from finmlsim import simulate as sim, metrics as met
import numpy as np

returns = sim.panel(n=260, n_assets=20, seed=6)
prediction, actual = returns[:-1], returns[1:]
rank_ic = np.mean([met.rank_ic(p, a) for p, a in zip(prediction, actual)])
weights = prediction / np.abs(prediction).sum(axis=1, keepdims=True)
strategy = (weights * actual).sum(axis=1)
print('rank IC:', round(rank_ic, 3))
print('turnover:', round(met.turnover(weights), 3))
print('gross Sharpe:', round(met.sharpe(strategy), 2))

## 2. Build the metrics from scratch
Translate each definition directly into array operations.

In [ ]:
rank = lambda x: np.argsort(np.argsort(x)).astype(float)
ics = [np.corrcoef(rank(p), rank(a))[0, 1] for p, a in zip(prediction, actual)]
scratch_turnover = np.abs(np.diff(weights, axis=0)).sum(axis=1).mean()
scratch_sharpe = np.sqrt(252) * strategy.mean() / strategy.std(ddof=1)
print('rank IC:', round(np.mean(ics), 3))
print('turnover:', round(scratch_turnover, 3))
print('gross Sharpe:', round(scratch_sharpe, 2))

## What to notice
Rank IC evaluates ordering, turnover describes how much the positions move, and Sharpe evaluates a return path. No one number substitutes for the others.